In [ ]:
import modules.data as d
import modules.utils as u
from pathlib import Path

#### init ####
dataset_dir = Path('/home/mv18gs/Documents/GitHub/pathway_model/datasets/')
device = u.Devices().auto_set_device(drop=['cuda:4'])

#### data ####
brca = d.TCGA(
    tcga_project = 'BRCA',
    tcga_dir = dataset_dir/'tcga',
    # type_col = 'sample_type',
    subtype_col = 'paper_BRCA_Subtype_PAM50',
    drop = ['Normal', 'Primary Tumor', 'Metastatic'],
    gene_name_path = dataset_dir/'other'/'name2ensg.csv',
    keep_noname = False,
)

kegg = d.KEGG(
    relation_filepath=dataset_dir/'other'/'relation_ohe.csv', 
    counts_data=brca,
)

dataset = d.GraphDataset(brca, kegg, kegg)
_batch = d.get_toy_databatch(dataset, device)

('cuda:4', 'NVIDIA A100-SXM4-80GB', 81043)
('cuda:2', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:3', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:7', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:5', 'NVIDIA A100-SXM4-80GB', 74619)
('cuda:1', 'NVIDIA A100-SXM4-80GB', 59033)
('cuda:0', 'NVIDIA A100-SXM4-80GB', 40950)
('cuda:6', 'NVIDIA A100-SXM4-80GB', 36533)

# #### Device() ####
# device = cuda:2

# #### KEGG() ####
# _orig_kwargs             5                        dict
# relation                 (75939, 19)              DataFrame
# ensg                     4373                     list
# pathway_labels           305                      list
# edge_index               (2, 32464)               Tensor (cuda:2)
# edge_attr                (32464, 16)              Tensor (cuda:2)
# edge_labels              16                       list
# pathway_index            (4373, 305)              Tensor (cuda:2)

# #### TCGA() ####
# _orig_kwargs             9                        dict
# counts_path            

In [2]:
from modules.layers import AttentionSetPooling, MultiheadSetPooling
from modules.model import MultiLatentModel
from modules.norm import LogCounts
from modules.train import MultiTrainer, MultiTrainerStage, Experiment, grid
from modules.trainers import ReconstrTrainer, ClassifTrainer
from modules.loss import NBLoss, KLDLoss, MultiLoss

import torch.nn as nn
from functools import partial

In [3]:
# multihead model
mh_model_template = partial(
    MultiLatentModel,
    dataset = dataset,
    embed_dim = 128,
    # head_dim = None (default)
    # num_heads = 1 (default) !!! handled in grid !!!
    method = 'node',

    # layers
    norm_class = LogCounts,
    encoder_class = nn.Linear,
    pooling_class = MultiheadSetPooling,
    mlp = False,
    variational = True,
    # out_module = nn.Linear (default)

    # layer params
    hidden_dims = 1,
    act_fn = nn.ReLU, 
    norm_fn = 'layer', 
    end_fn = False,

    # kwargs
    norm_kwargs = {'libnorm':True, 'znorm':True, 'learnable':True}
    # pooling_kwargs = None (default) !!! handled in grid !!!
)

mh_model_grid = grid(
    mh_model_template,
    num_heads = [1, 2, 4, 8],
    pooling_kwargs = [{'aggregate':'concat'}, {'aggregate':'mean'}]
)

In [4]:
# attention setpooling model
as_model_template = partial(
    MultiLatentModel,
    dataset = dataset,
    embed_dim = 128,
    # head_dim = None, (default)
    # num_heads = 1, (default)
    method = 'node',

    # layers
    norm_class = LogCounts,
    encoder_class = nn.Linear,
    pooling_class = AttentionSetPooling,
    mlp = False,
    variational = True,
    # out_module = nn.Linear, (default)

    # layer params
    hidden_dims = 1,
    act_fn = nn.ReLU, 
    norm_fn = 'layer', 
    end_fn = False,

    # kwargs
    norm_kwargs = {'libnorm':True, 'znorm':True, 'learnable':True},
    # pooling_kwargs = None, (default)
)

as_model_grid = grid(
    as_model_template,
)

In [5]:
# trainer
ae_stage = MultiTrainerStage(
    name = 'ae',
    train = ['encoder', 'latent_ae', 'decoder'],
    trainer = ReconstrTrainer(
        lr=1e-3, 

        trainer_norm_class=LogCounts,
        early_stop=True,
        stop_metric='mae',

        kw_keys=('x','x_pred','theta','z_mu_ae','z_logvar_ae'),
        metric_keys={'pred':'x_pred', 'target':'x'},
        
        loss_class=MultiLoss,
        loss_kwargs = {
        'loss_classes': [NBLoss, KLDLoss],
        'loss_weights': [1.0, 1e-4],
        'loss_inputs': [
            {'x':'x', 'mu':'x_pred', 'theta':'theta'},
            {'mu':'z_mu_ae', 'logvar':'z_logvar_ae'}
        ]
    },
    )
)

cl_stage = MultiTrainerStage(
    name = 'cl',
    train = ['latent_cl', 'classifier'],
    trainer = ClassifTrainer(
        lr=1e-3, 

        early_stop=True,
        stop_metric='accuracy',
        stop_kwargs={'mode':'max'},

        kw_keys={'input':'y_logits', 'target':'y'},
        metric_keys={'pred':'input', 'target':'target'},
        loss_class=nn.CrossEntropyLoss,
        # loss_kwargs={'label_smoothing':0.1}
    )
)

trainer = MultiTrainer(
    stages = [ae_stage, cl_stage]
)

In [6]:
# expt
expt = Experiment(
    num_trials=30, # scVI: 5 trials
    num_epochs=300, # scVI: 200 epochs
    dataset=dataset,
    device=device,
    batch_size=128
)

# expt.add_config(
#     name = 'multitrainer_test',
#     model = model,
#     trainer = trainer,
# )

# multihead model grid
expt.add_grid(
    model_grid = mh_model_grid,
    trainer_grid = trainer,
    prefix='mh'
)

# attention setpooling model grid
expt.add_grid(
    model_grid = as_model_grid,
    trainer_grid = trainer,
    prefix='as'
)

In [7]:
for i,j in enumerate(expt.configs.keys()):
    print(f'{i}: {j}')

0: mh_numheads1_aggregateConcat
1: mh_numheads1_aggregateMean
2: mh_numheads2_aggregateConcat
3: mh_numheads2_aggregateMean
4: mh_numheads4_aggregateConcat
5: mh_numheads4_aggregateMean
6: mh_numheads8_aggregateConcat
7: mh_numheads8_aggregateMean
8: as_config


In [ ]:
expt.run_experiment(
    'grid_5b_multilatent_multihead_v_setpool',
    report_metrics=['loss','kld','nb','mae'],
    save_csv=True,
    save_params=True,
    save_values=True,
    verbose=False,
)

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/9 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

KeyboardInterrupt: 